In [0]:
%python
# Widgets del proyecto
dbutils.widgets.removeAll()

In [0]:
%sql
-- Widgets del proyecto
create widget text storageName default "sanvarkim";

In [0]:

#Funciones requeridas
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:

#Definicion de widgets
dbutils.widgets.text("catalogo", "sanvar_kim")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:

#Parametros
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
%sql
-- Uso de schema a utilizar
USE CATALOG sanvar_kim;
USE SCHEMA bronze;

In [0]:
df_bronze_persona = spark.table(f"{catalogo}.{esquema_source}.bronze_persona")
df_bronze_alquiler = spark.table(f"{catalogo}.{esquema_source}.bronze_alquiler")
df_bronze_apartamento = spark.table(f"{catalogo}.{esquema_source}.bronze_apartamento")

df_resultado = df_bronze_persona.join(
    df_bronze_alquiler,
    df_bronze_persona["Numero_de_Identificacion"] == df_bronze_alquiler["Numero_de_Identificacion2"],
    "inner"
).join(
    df_bronze_apartamento,
    df_bronze_apartamento["No"] == df_bronze_alquiler["Id_apartamento"],
    "inner"
).select(
    df_bronze_alquiler["No"],
    df_bronze_persona["Numero_de_Identificacion"],
    df_bronze_persona["Nombre_persona"],
    df_bronze_persona["Apellido_persona"],
    df_bronze_persona["Fecha_de_Nacimiento"],
    df_bronze_persona["Sexo"],
    df_bronze_persona["Edad"],
    df_bronze_persona["Peso"],
    df_bronze_persona["Direccion"],
    df_bronze_apartamento["No"],
    df_bronze_apartamento["Habitaciones"],
    df_bronze_apartamento["Cochera"],
    df_bronze_apartamento["Servicios"],
    df_bronze_apartamento["Ubicacion"],
    df_bronze_alquiler["Estado"],
    df_bronze_alquiler["Fecha_inicio"],
    df_bronze_alquiler["Fecha_fin"]
).filter(df_bronze_alquiler["Estado"] == "A")

#display(df_resultado)

In [0]:
#Escribir datos completos en tabla silver
df_resultado.write.mode("overwrite").insertInto("sanvar_kim.silver.detalle_alquileres")

In [0]:

#Confirmar datos incertados
display(
    spark.table("sanvar_kim.silver.detalle_alquileres")
)